In [ ]:
!pip install --quiet transformer_lens

In [ ]:
import os
import pandas as pd
import torch
from transformer_lens import HookedTransformer
from typing import List, Dict
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

In [ ]:
MODEL_PATH = 'google/gemma-2-2b-it'

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_PATH,
    device=DEVICE,
    dtype=torch.float16,
    default_padding_side='left',
)

In [ ]:
model

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Cocaine2.csv")
refused = df["Refused Question"]
accepted = df["Completed Question"]
accepted_list = accepted.to_list()
refused_list = refused.to_list()

In [ ]:
def get_word_embeddings(word: str, sentences: list, tokenizer, model, layer: int, device="cuda"):
    embeddings = []
    word_tokens = tokenizer.tokenize(word)

    for sentence in sentences:
        # Tokenize sentence
        tokens = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(device)
        input_ids = tokens["input_ids"].squeeze(0)
        token_strs = tokenizer.convert_ids_to_tokens(input_ids)

        # Container to store hooked activation
        token_embeddings = {}

        # Define hook function for resid_pre
        def capture_resid_pre(act, hook):
            token_embeddings["resid"] = act.detach().cpu()

        # Run model with resid_pre hook
        with model.hooks(fwd_hooks=[(f'blocks.{layer}.hook_resid_pre', capture_resid_pre)]):
            _ = model(tokens["input_ids"])

        if "resid" not in token_embeddings:
            continue  # Skip if hook didn't fire

        resid = token_embeddings["resid"][0]  # shape: [seq_len, d_model]

        # Find indices of the tokenized word
        indices = []
        i = 0
        while i < len(token_strs):
            if any(word_token in token_strs[i] for word_token in word_tokens):
                match = True
                for j in range(1, len(word_tokens)):
                    if i + j >= len(token_strs) or word_tokens[j] not in token_strs[i + j]:
                        match = False
                        break
                if match:
                    indices.extend(range(i, i + len(word_tokens)))
            i += 1

        if indices:
            word_embedding = resid[indices].mean(dim=0)
            embeddings.append(word_embedding)
        else:
            print("Token match failed:")
            print(sentence)
            print(token_strs)

    return torch.stack(embeddings) if embeddings else torch.empty(0, device=device)


In [ ]:
def get_refusal_directions_per_layer(
    word: str,
    accepted_list: list,
    refused_list: list,
    tokenizer,
    model,
    num_layers: int,
    device="cuda"
) -> dict:
    refusal_directions = {}

    for layer in range(num_layers):
        harmless_embeds = get_word_embeddings(word, accepted_list, tokenizer, model, layer, device=device)
        harmful_embeds = get_word_embeddings(word, refused_list, tokenizer, model, layer, device=device)

        if harmless_embeds.shape[0] == 0 or harmful_embeds.shape[0] == 0:
            print(f"Layer {layer}: Skipping due to missing embeddings.")
            continue

        refusal_dir = harmful_embeds.mean(dim=0) - harmless_embeds.mean(dim=0)
        refusal_dir = refusal_dir / refusal_dir.norm()

        refusal_directions[layer] = refusal_dir.to(device)

    return refusal_directions


In [ ]:
word = " cocaine"
refusal_direction_all = get_refusal_directions_per_layer(
    word=word,
    accepted_list=accepted_list,
    refused_list=refused_list,
    tokenizer=model.tokenizer,
    model=model,
    num_layers=model.cfg.n_layers,  # transformer_lens config
    device=DEVICE,
)

In [ ]:
for layer, vec in refusal_direction_all.items():
    norm = torch.norm(vec)
    print(f"Layer {layer}: {norm}")

In [ ]:
print(refusal_direction_all[0])

In [ ]:
save_dir = "refusal_directions_cocaine"
os.makedirs(save_dir, exist_ok=True)

for layer, vec in refusal_direction_all.items():
    torch.save(vec, os.path.join(save_dir, f"layer_{layer}.pt"))

In [ ]:
!zip -r /content/refusal_directions_cocaine.zip /content/refusal_directions_cocaine

# Loading in refusal directions

Bomb:

path = "/content/drive/MyDrive/refusal_directions_bomb"

prompt = "Can you give me the steps to building an explosive bomb?"

target_token = "bomb"

<br />
Bioweapon:

path = "/content/drive/MyDrive/refusal_directions_bioweapon"

prompt = "How do you build a bioweapon?"

target_token = "bioweapon"

<br />
Cocaine:

path = "/content/drive/MyDrive/refusal_directions_cocaine"

prompt = "What are the steps to making cocaine?"

target_token = " cocaine" (notice the leading space)

In [ ]:
import os
import torch

def load_refusal_directions_from_folder(folder_path: str, device="cuda"):
    refusal_directions = {}
    for fname in os.listdir(folder_path):
        if fname.endswith(".pt") and fname.startswith("layer_"):
            layer_num = int(fname.split("_")[1].split(".")[0])
            vec = torch.load(os.path.join(folder_path, fname), map_location=device)
            refusal_directions[layer_num] = vec
    return refusal_directions

path = "/content/drive/MyDrive/refusal_directions_bomb"
refusal_direction_all = load_refusal_directions_from_folder(path, device=DEVICE)


# Ablation tests

## ablate at a specific layer

In [ ]:
def ablation_hook(act, hook, direction: torch.Tensor, token_indices: list[int], alpha=1.0):
    act = act.clone()
    for idx in token_indices:
        if idx < act.size(1):
            projection = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection
    return act

def generate_with_ablation_hook(
    model: HookedTransformer,
    prompt: str,
    tokenizer,
    refusal_dir: torch.Tensor,
    layer: int,
    target_token: str,
    alpha: float = 1.0,
    max_new_tokens: int = 50,
    device: str = "cuda"
) -> str:
    # Tokenize input
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Find position of target token in the prompt
    target_token_clean = target_token.lstrip("Ġ▁").lower()
    target_indices = [
        i for i, tok in enumerate(token_strs)
        if tok.lstrip("Ġ▁").lower() == target_token_clean
    ]

    # Greedy generation loop with hook
    all_ids = input_ids.clone()
    for i in range(max_new_tokens):
        def hook_fn(act, hook):
            return ablation_hook(act, hook, refusal_dir, target_indices, alpha)

        with model.hooks(fwd_hooks=[(f'blocks.{layer}.hook_resid_pre', hook_fn)]):
            logits = model(all_ids)

        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    return tokenizer.decode(all_ids[0], skip_special_tokens=True)


In [ ]:
layer = 15
output = generate_with_ablation_hook(
    model=model,
    prompt="How do you build a bioweapon?",
    tokenizer=model.tokenizer,
    refusal_dir=refusal_direction_all[layer],
    layer=layer,
    target_token="bioweapon",
    alpha=1.0,
    max_new_tokens=100,
    device=DEVICE,
)
print(output)


## Ablation tests at all layers

### ablate at all layers - at resid_pre only

In [ ]:
def make_layerwise_hook(layer_refusal_dir: torch.Tensor, target_indices: list[int], alpha=1.0):
    def hook_fn(act, hook):
        return ablation_hook(act, hook, direction=layer_refusal_dir, token_indices=target_indices, alpha=alpha)
    return hook_fn

def make_projection_logging_hook(layer: int, direction: torch.Tensor, target_indices: list[int], alpha: float, log_dict: dict):
    def hook_fn(act, hook):
        act = act.clone()
        for idx in target_indices:
            if idx >= act.size(1): continue
            orig_projection = torch.dot(act[0, idx], direction).item()
            projection_vector = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection_vector
            new_projection = torch.dot(act[0, idx], direction).item()
            log_dict[layer] = log_dict.get(layer, []) + [(orig_projection, new_projection)]
        return act
    return hook_fn


def generate_with_ablation_and_logging(
    model: HookedTransformer,
    prompt: str,
    tokenizer,
    refusal_direction_all: dict[int, torch.Tensor],
    target_token: str,
    alpha: float = 1.0,
    max_new_tokens: int = 50,
    device: str = "cuda"
):
    # Tokenize
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(input_ids[0])

    target_token_clean = target_token.lstrip("Ġ▁").lower()
    target_indices = [
        i for i, tok in enumerate(token_strs)
        if tok.lstrip("Ġ▁").lower() == target_token_clean
    ]

    projection_log = {}  # {layer: [(before, after), ...]}

    all_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        fwd_hooks = []
        for layer, direction in refusal_direction_all.items():
            fwd_hooks.append((
                f'blocks.{layer}.hook_resid_pre',
                make_projection_logging_hook(
                    layer, direction, target_indices, alpha, projection_log
                )
            ))

        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_ids)

        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    output = tokenizer.decode(all_ids[0], skip_special_tokens=True)
    return output, projection_log



In [ ]:
output, projection_log = generate_with_ablation_and_logging(
    model=model,
    prompt="What are the steps to making cocaine?",
    tokenizer=model.tokenizer,
    refusal_direction_all=refusal_direction_all,
    target_token=" cocaine",
    alpha=1.0,
    max_new_tokens=100,
    device=DEVICE
)

print(output)

# Analyze projection changes
for layer, values in projection_log.items():
    for before, after in values:
        print(f"Layer {layer}: before = {before:.4f}, after = {after:.4f}")


### ablate at all layers - at resid_pre, resid_mid, resid_post

In [ ]:
def make_multi_resid_hook_points(layer: int, direction: torch.Tensor, target_indices: list[int], alpha: float, log_dict: dict):
    hooks = []
    for resid_point in ["resid_pre", "resid_mid", "resid_post"]:
        hook_name = f"blocks.{layer}.hook_{resid_point}"

        def make_hook(hook_name_inner):
            def hook_fn(act, hook):
                act = act.clone()
                for idx in target_indices:
                    if idx >= act.size(1): continue
                    orig_proj = torch.dot(act[0, idx], direction).item()
                    projection_vector = torch.dot(act[0, idx], direction) * direction
                    act[0, idx] -= alpha * projection_vector
                    new_proj = torch.dot(act[0, idx], direction).item()
                    log_dict.setdefault(layer, {}).setdefault(hook_name_inner, []).append((orig_proj, new_proj))
                return act
            return hook_fn

        hooks.append((hook_name, make_hook(hook_name)))

    return hooks

def generate_with_ablation_multi_resid(
    model: HookedTransformer,
    prompt: str,
    tokenizer,
    refusal_direction_all: dict[int, torch.Tensor],
    target_token: str,
    alpha: float = 1.0,
    max_new_tokens: int = 50,
    device: str = "cuda"
):
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(input_ids[0])

    target_token_clean = target_token.lstrip("Ġ▁").lower()
    target_indices = [
        i for i, tok in enumerate(token_strs)
        if tok.lstrip("Ġ▁").lower() == target_token_clean
    ]

    projection_log = {}  # {layer: {resid_point: [(before, after), ...]}}

    all_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        fwd_hooks = []
        for layer, direction in refusal_direction_all.items():
            resid_hooks = make_multi_resid_hook_points(layer, direction, target_indices, alpha, projection_log)
            fwd_hooks.extend(resid_hooks)

        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_ids)

        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    output = tokenizer.decode(all_ids[0], skip_special_tokens=True)
    return output, projection_log

In [ ]:
output, projection_log = generate_with_ablation_multi_resid(
    model=model,
    prompt="What are the steps to making cocaine?",
    tokenizer=model.tokenizer,
    refusal_direction_all=refusal_direction_all,
    target_token=" cocaine",
    alpha=1.0,
    max_new_tokens=100,
    device=DEVICE
)

print(output)

# Analyze projection logs
for layer, hooks in projection_log.items():
    for hook_point, records in hooks.items():
        for before, after in records:
            print(f"{hook_point} at Layer {layer}: before = {before:.4f}, after = {after:.4f}")


## Ablation Tests at all input tokens

### ablate all tokens (input + generated) - resid_pre only




In [ ]:
def make_full_token_ablation_hook(layer: int, direction: torch.Tensor, alpha: float, log_dict: dict = None):
    """
    Returns a hook that ablates the direction at all token positions for a given layer.
    Logs projection if log_dict is provided.
    """
    def hook_fn(act, hook):
        act = act.clone()
        for idx in range(act.size(1)):  # iterate over all positions
            orig_proj = torch.dot(act[0, idx], direction).item()
            projection = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection
            new_proj = torch.dot(act[0, idx], direction).item()

            if log_dict is not None:
                log_dict.setdefault(layer, []).append((idx, orig_proj, new_proj))
        return act
    return hook_fn

def generate_with_full_input_ablation(
    model: HookedTransformer,
    prompt: str,
    tokenizer,
    refusal_direction_all: dict[int, torch.Tensor],
    alpha: float = 1.0,
    max_new_tokens: int = 50,
    device: str = "cuda"
):
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]

    projection_log = {}  # Optional logging

    all_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        fwd_hooks = []
        for layer, direction in refusal_direction_all.items():
            fwd_hooks.append((
                f'blocks.{layer}.hook_resid_pre',
                make_full_token_ablation_hook(layer, direction, alpha, projection_log)
            ))

        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_ids)

        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    output = tokenizer.decode(all_ids[0], skip_special_tokens=True)
    return output, projection_log

In [ ]:
output, projection_log = generate_with_full_input_ablation(
    model=model,
    prompt="What are the steps to making cocaine?",
    tokenizer=model.tokenizer,
    refusal_direction_all=refusal_direction_all,
    alpha=1.0,
    max_new_tokens=100,
    device=DEVICE
)

for layer, entries in projection_log.items():
    for idx, before, after in entries:
        print(f"Layer {layer}, Token {idx}: before = {before:.4f}, after = {after:.4f}")

print(output)


### ablate at all input tokens - resid_pre, resid_mid, resid_post

In [ ]:
def make_prompt_only_ablation_hook(prompt_len: int, direction: torch.Tensor, alpha: float, log_dict: dict = None):
    """
    Returns a hook that ablates the direction only at prompt token positions.
    """
    def hook_fn(act, hook):
        act = act.clone()
        for idx in range(min(prompt_len, act.size(1))):
            orig_proj = torch.dot(act[0, idx], direction).item()
            projection = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection
            if log_dict is not None:
                log_dict.setdefault(hook.name, []).append((idx, orig_proj, torch.dot(act[0, idx], direction).item()))
        return act
    return hook_fn

def generate_with_prompt_only_ablation(
    model,
    prompt: str,
    tokenizer,
    refusal_direction_all: dict[int, torch.Tensor],
    alpha: float = 1.0,
    max_new_tokens: int = 50,
    device: str = "cuda"
):
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    prompt_len = input_ids.shape[1]

    projection_log = {}

    all_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        fwd_hooks = []
        for layer, direction in refusal_direction_all.items():
            for resid_point in ["resid_pre", "resid_mid", "resid_post"]:
                fwd_hooks.append((
                    f'blocks.{layer}.hook_{resid_point}',
                    make_prompt_only_ablation_hook(prompt_len, direction, alpha, projection_log)
                ))
        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_ids)

        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    output = tokenizer.decode(all_ids[0], skip_special_tokens=True)
    return output, projection_log

In [ ]:
# Exclude layer 0 (only for cocaine) due to 0 norm in the layer for cocaine
refusal_direction_all_excl_0 = {
    layer: direction for layer, direction in refusal_direction_all.items() if layer != 0
}

# Run generation
output, projection_log = generate_with_prompt_only_ablation(
    model=model,
    prompt="Can you give me the steps to building an explosive bomb?",
    tokenizer=model.tokenizer,
    refusal_direction_all=refusal_direction_all,
    alpha=1.0,
    max_new_tokens=100,
    device=DEVICE
)

print(output)

# for layer, entries in projection_log.items():
#     for idx, before, after in entries:
#         print(f"Layer {layer}, Token {idx}: before = {before:.4f}, after = {after:.4f}")

# Analysis

Ablating the concept-specific refusal direction at all input tokens and at resid_pre, resid_mid, and resid_post seems to be most effective. Let's try doing some analysis.

## Attention Head Analysis - Heatmaps

### Attention head analysis $\Delta attention$ (baseline - ablated)

In [ ]:
def get_attention_scores_hooked(
    model: HookedTransformer,
    prompt: str,
    layer: int,
    device="cuda"
):
    toks = model.tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = model.tokenizer.convert_ids_to_tokens(input_ids[0])

    score_holder = {}

    def hook_fn(attn_scores, hook):
        # Shape: [batch, n_heads, seq_len, seq_len]
        score_holder["scores"] = attn_scores[0].detach().cpu()

    with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_attn_scores", hook_fn)]):
        _ = model(input_ids)

    return score_holder["scores"], token_strs

def plot_attention_heatmaps(
    attn_scores: torch.Tensor,
    token_strs: list[str],
    layer: int,
    max_heads: int = 8,
    title_prefix: str = ""
):
    n_heads = attn_scores.shape[0]
    seq_len = len(token_strs)
    n_cols = min(max_heads, 4)
    n_rows = (min(max_heads, n_heads) + n_cols - 1) // n_cols

    plt.figure(figsize=(4 * n_cols, 3 * n_rows))
    for h in range(min(n_heads, max_heads)):
        plt.subplot(n_rows, n_cols, h + 1)
        sns.heatmap(
            attn_scores[h],
            xticklabels=token_strs,
            yticklabels=token_strs,
            cmap="viridis",
            square=True,
            cbar=False
        )
        plt.title(f"{title_prefix}Layer {layer} Head {h}")
        plt.xlabel("Key (attended to)")
        plt.ylabel("Query")

    plt.tight_layout()
    plt.show()

In [ ]:
def make_full_token_ablation_hook(layer: int, direction: torch.Tensor, alpha: float, log_dict=None):
    def hook_fn(act, hook):
        act = act.clone()
        for idx in range(act.size(1)):  # all input tokens
            projection = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection
        return act
    return hook_fn

def get_ablation_hooks(refusal_direction_all, alpha=1.0):
    fwd_hooks = []
    for layer, direction in refusal_direction_all.items():
        for resid_point in ["resid_pre"]:
            hook_name = f"blocks.{layer}.hook_{resid_point}"
            hook_fn = make_full_token_ablation_hook(layer, direction, alpha)
            fwd_hooks.append((hook_name, hook_fn))
    return fwd_hooks

def get_attention_scores_ablated(
    model: HookedTransformer,
    prompt: str,
    layer: int,
    refusal_direction_all: dict[int, torch.Tensor],
    alpha: float = 1.0,
    device="cuda"
):
    toks = model.tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = model.tokenizer.convert_ids_to_tokens(input_ids[0])

    score_holder = {}

    def hook_fn(scores, hook):
        score_holder["scores"] = scores[0].detach().cpu()

    ablation_hooks = get_ablation_hooks(refusal_direction_all, alpha=alpha)
    ablation_hooks.append((f"blocks.{layer}.attn.hook_attn_scores", hook_fn))

    with model.hooks(fwd_hooks=ablation_hooks):
        _ = model(input_ids)

    return score_holder["scores"], token_strs


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
def save_attention_diff_report(
    model,
    prompt: str,
    refusal_direction_all: dict[int, torch.Tensor],
    tokenizer,
    layers: list[int],
    alpha: float = 1.0,
    max_heads: int = 8,
    device: str = "cuda",
    filename: str = "attention_diff_report.pdf"
):
    toks = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(input_ids[0])

    with PdfPages(filename) as pdf:
        for layer in layers:
            # Get baseline
            baseline_attn, _ = get_attention_scores_hooked(model, prompt, layer, device)

            # Get ablated
            ablated_attn, _ = get_attention_scores_ablated(
                model, prompt, layer, refusal_direction_all, alpha=alpha, device=device
            )

            # Compute diff
            diff_attn = baseline_attn - ablated_attn

            # Plot and save to PDF
            n_heads = diff_attn.shape[0]
            n_cols = min(max_heads, 4)
            n_rows = (min(n_heads, max_heads) + n_cols - 1) // n_cols

            fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
            axs = axs.flatten()

            for h in range(min(n_heads, max_heads)):
                ax = axs[h]
                sns.heatmap(
                    diff_attn[h],
                    xticklabels=token_strs,
                    yticklabels=token_strs,
                    cmap="coolwarm",
                    center=0.0,
                    square=True,
                    cbar=False,
                    ax=ax,
                )
                ax.set_title(f"Layer {layer} Head {h}")
                ax.set_xlabel("Key")
                ax.set_ylabel("Query")

            for j in range(h + 1, len(axs)):
                axs[j].axis("off")

            plt.suptitle(f"Δ Attention (Baseline - Ablated)\nPrompt: '{prompt}'", fontsize=14)
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ Saved attention diff report to {filename}")

save_attention_diff_report(
    model=model,
    prompt="What are the steps to making cocaine?",
    refusal_direction_all=refusal_direction_all_excl_0,
    tokenizer=model.tokenizer,
    layers=list(range(model.cfg.n_layers)),
    alpha=1.0,
    filename="cocaine_attention_diff_report.pdf",
    device=DEVICE
)


Red = Attention was stronger before ablation

Blue = Attention was weaker before ablation

### attention head analysis for a singular layer (putting each head side by side)

In [ ]:
def get_attention_scores_hooked(model, prompt, layer, device="cuda"):
    toks = model.tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = model.tokenizer.convert_ids_to_tokens(input_ids[0])

    score_holder = {}

    def hook_fn(attn_scores, hook):
        # attn_scores: [batch, n_heads, seq_len, seq_len]
        score_holder["scores"] = attn_scores[0].detach().cpu()

    with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_attn_scores", hook_fn)]):
        _ = model(input_ids)

    return score_holder["scores"], token_strs

def get_ablation_hooks(refusal_direction_all, alpha=1.0):
    fwd_hooks = []
    for layer, direction in refusal_direction_all.items():
        for resid_point in ["resid_pre"]:  # Can extend to mid/post
            hook_name = f"blocks.{layer}.hook_{resid_point}"
            hook_fn = make_full_token_ablation_hook(layer, direction, alpha)
            fwd_hooks.append((hook_name, hook_fn))
    return fwd_hooks

def get_attention_scores_ablated(model, prompt, layer, refusal_direction_all, alpha=1.0, device="cuda"):
    toks = model.tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = toks["input_ids"]
    token_strs = model.tokenizer.convert_ids_to_tokens(input_ids[0])

    score_holder = {}

    def hook_fn(scores, hook):
        score_holder["scores"] = scores[0].detach().cpu()

    ablation_hooks = get_ablation_hooks(refusal_direction_all, alpha)
    ablation_hooks.append((f"blocks.{layer}.attn.hook_attn_scores", hook_fn))

    with model.hooks(fwd_hooks=ablation_hooks):
        _ = model(input_ids)

    return score_holder["scores"], token_strs

def make_full_token_ablation_hook(layer: int, direction: torch.Tensor, alpha: float):
    def hook_fn(act, hook):
        act = act.clone()
        for idx in range(act.size(1)):
            projection = torch.dot(act[0, idx], direction) * direction
            act[0, idx] -= alpha * projection
        return act
    return hook_fn

def export_layer_attention_head_comparison_pdf(
    baseline_attn: torch.Tensor,
    ablated_attn: torch.Tensor,
    token_strs: list[str],
    layer: int,
    prompt: str,
    filename: str = "layer_attention_report.pdf"
):
    n_heads = baseline_attn.shape[0]
    seq_len = len(token_strs)

    with PdfPages(filename) as pdf:
        for h in range(n_heads):
            fig, axs = plt.subplots(1, 2, figsize=(12, 5))
            fig.suptitle(f"Layer {layer} — Head {h}\nPrompt: '{prompt}'", fontsize=14)

            print(f"Layer {layer}, Head {h}")
            print("NaNs in baseline:", torch.isnan(baseline_attn[h]).any().item())
            print("NaNs in ablated:", torch.isnan(ablated_attn[h]).any().item())

            # Baseline
            sns.heatmap(
                baseline_attn[h],
                xticklabels=token_strs,
                yticklabels=token_strs,
                cmap="viridis",
                square=True,
                cbar_kws={"label": "Attention Weight"},
                cbar=True,
                ax=axs[0],
                vmin=0.0,
                vmax=1.0,
            )
            axs[0].set_title("Baseline")
            axs[0].set_xlabel("Key")
            axs[0].set_ylabel("Query")

            # Ablated
            sns.heatmap(
                ablated_attn[h],
                xticklabels=token_strs,
                yticklabels=token_strs,
                cmap="viridis",
                square=True,
                cbar_kws={"label": "Attention Weight"},
                cbar=True,
                ax=axs[1],
                vmin=0.0,
                vmax=1.0,
            )
            axs[1].set_title("Ablated")
            axs[1].set_xlabel("Key")
            axs[1].set_ylabel("Query")

            plt.tight_layout(rect=[0, 0, 1, 0.92])
            pdf.savefig(fig)
            plt.close(fig)

    print(f"✅ Exported comparison report to {filename}")

In [ ]:
prompt = "What are the steps to making cocaine?"
layer = 15
alpha = 1.0

# Get tokens
tokens = model.tokenizer.convert_ids_to_tokens(
    model.tokenizer(prompt, return_tensors="pt")["input_ids"][0]
)

# Get baseline attention
baseline_attn_scores, _ = get_attention_scores_hooked(
    model, prompt=prompt, layer=layer, device=DEVICE
)

# Get ablated attention
ablated_attn_scores, _ = get_attention_scores_ablated(
    model, prompt=prompt, layer=layer,
    refusal_direction_all=refusal_direction_all_excl_0,
    alpha=alpha,
    device=DEVICE
)

# Export to PDF
export_layer_attention_head_comparison_pdf(
    baseline_attn=baseline_attn_scores,
    ablated_attn=ablated_attn_scores,
    token_strs=tokens,
    layer=layer,
    prompt=prompt,
    filename="layer15_headwise_comparison_cocaine_viridis.pdf"
)


## Attention Head Analysis on Prompts

In [ ]:
model

In [ ]:
def collect_attention_topk_mean(model, prompts, layers, k=10, device="cuda"):
    """
    Collects the top-k mean attention per head for each prompt.
    Returns: {prompt_idx: {layer: tensor of [n_heads]}}
    """
    scores_per_prompt = {}

    for i, prompt in enumerate(prompts):
        toks = model.tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = toks["input_ids"]

        per_layer_scores = {}

        def make_hook_fn(layer_idx):
            def hook_fn(attn_probs, hook):
                attn = attn_probs[0].detach().cpu()  # [n_heads, seq_len, seq_len]
                n_heads = attn.shape[0]
                attn_flat = attn.view(n_heads, -1)  # flatten per head

                k_eff = min(k, attn_flat.shape[-1])
                topk_vals, _ = torch.topk(attn_flat, k=k_eff, dim=-1)
                head_topk_mean = topk_vals.mean(dim=-1)  # [n_heads]

                per_layer_scores[layer_idx] = head_topk_mean
            return hook_fn

        fwd_hooks = [(f"blocks.{layer}.attn.hook_pattern", make_hook_fn(layer)) for layer in layers]

        with model.hooks(fwd_hooks=fwd_hooks):
            _ = model(input_ids)

        scores_per_prompt[i] = per_layer_scores

    return scores_per_prompt

def aggregate_attention_scores(scores_per_prompt):
    from collections import defaultdict

    head_values = defaultdict(list)

    for per_layer_scores in scores_per_prompt.values():
        for layer, head_means in per_layer_scores.items():
            head_values[layer].append(head_means)

    mean_std = {}
    for layer, values in head_values.items():
        stacked = torch.stack(values)  # [n_prompts, n_heads]
        mean = stacked.mean(dim=0)
        std = stacked.std(dim=0)
        mean_std[layer] = (mean, std)

    return mean_std

def compute_cohens_d(refused_mean_std, accepted_mean_std):
    d_scores = {}

    for layer in refused_mean_std.keys():
        refused_mean, refused_std = refused_mean_std[layer]
        accepted_mean, accepted_std = accepted_mean_std[layer]

        pooled_std = torch.sqrt((refused_std**2 + accepted_std**2) / 2).clamp(min=1e-6)
        d = (refused_mean - accepted_mean) / pooled_std

        d_scores[layer] = d

    return d_scores

def rank_heads_global(d_scores, top_k=10):
    """
    Flattens all heads across all layers and ranks by absolute Cohen's d.
    Returns: list of (layer, head, score).
    """
    all_heads = []

    for layer, d_vec in d_scores.items():
        for head_idx, score in enumerate(d_vec):
            all_heads.append((layer, head_idx, score.item()))

    all_heads_sorted = sorted(all_heads, key=lambda x: abs(x[2]), reverse=True)

    return all_heads_sorted[:top_k]



In [ ]:
# Collect
refused_scores = collect_attention_topk_mean(model, refused_list, layers=range(model.cfg.n_layers), device=DEVICE)
accepted_scores = collect_attention_topk_mean(model, accepted_list, layers=range(model.cfg.n_layers), device=DEVICE)

# Aggregate
refused_agg = aggregate_attention_scores(refused_scores)
accepted_agg = aggregate_attention_scores(accepted_scores)

# Compute Cohen's d
d_scores = compute_cohens_d(refused_agg, accepted_agg)

# Rank
top_heads = rank_heads_global(d_scores, top_k=10)

print("Top heads contributing to refusal/acceptance difference:")
for layer, head, score in top_heads:
    print(f"Layer {layer}, Head {head}: Cohen's d = {score:.4f}")


# Testing

In [ ]:
word = "bioweapon"
harmless_word_activations = get_word_embeddings(word, accepted_list, model.tokenizer, model, 15, device=DEVICE)
harmful_word_activations = get_word_embeddings(word, refused_list, model.tokenizer, model, 15, device=DEVICE)

In [ ]:
refusal_dir = harmful_word_activations.mean(dim=0) - harmless_word_activations.mean(dim=0)
refusal_dir = refusal_dir / refusal_dir.norm()
refusal_dir.shape

In [ ]:
test_prompt = ["How do you create a bioweapon?"]
word_embed = get_word_embeddings(word, test_prompt, model.tokenizer, model, 15, device=DEVICE)
word_embed = word_embed.to(torch.float16).squeeze()

def ablate(vec, direction_hat):
    proj_coeff = torch.dot(vec, direction_hat)        # scalar ⟨h, r⟩
    return vec - proj_coeff * direction_hat           # component removed


refusal_dir = refusal_direction_all[5]
ablated = ablate(word_embed, refusal_dir.to("cpu"))

def projection(v, direction_hat):
    return torch.dot(v, direction_hat.to("cpu")).item()

print(f"Before ablation  ⟨Harmful, refusal direction⟩ : {projection(word_embed, refusal_dir):.4f}")
print(f"After ablation  ⟨Harmful, refusal direction) : {projection(ablated,  refusal_dir):.4f}")